In [ ]:
# --- 1. INSTALAÇÃO ---
print("Instalando bibliotecas...")
!pip install geopandas folium shapely gspread google-auth google-auth-oauthlib google-auth-httplib2 ipynbname duckdb boto3 botocore unidecode geopy fuzzywuzzy python-Levenshtein awscli -q
print("Bibliotecas instaladas.")

# Configuração AWS
!aws configure set s3.signature_version s3v4
!aws configure set region us-west-2

# --- 2. IMPORTAÇÕES ---
import os
import re
import time
import datetime
import math
import duckdb
import requests
import pandas as pd
import geopandas as gpd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from io import BytesIO
from shapely.geometry import Point, Polygon, mapping
from shapely import wkb
from unidecode import unidecode
from fuzzywuzzy import fuzz
import folium
from branca.element import MacroElement
from jinja2 import Template
from google.colab import userdata, drive, auth as colab_auth
from google.auth import default as google_auth_default
import gspread
import urllib3
import numpy as np
import warnings

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- 3. CONFIGURAÇÃO ---
try:
    drive.mount('/content/drive')
    base_path_secret = userdata.get('DRIVE_SAVE_PATH') if 'DRIVE_SAVE_PATH' in userdata else "/content/drive/MyDrive/Colab_Pipelines/Maia_Saude_V74/"
    DRIVE_SAVE_PATH = base_path_secret
    if not os.path.exists(DRIVE_SAVE_PATH): os.makedirs(DRIVE_SAVE_PATH)
except:
    DRIVE_SAVE_PATH = "/content/"

# Secrets & Email Config
try:
    GMAIL_USER = userdata.get('GMAIL_USER')
    GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
    email_list_raw = userdata.get('DESTINATION_EMAIL_LIST')
    if "," in email_list_raw:
        DESTINATION_EMAIL_LIST = [e.strip() for e in email_list_raw.split(',')]
    else:
        DESTINATION_EMAIL_LIST = [email_list_raw.strip()]
except Exception as e:
    print(f"⚠️ Aviso: Secrets de email não configurados corretamente. ({e})")
    GMAIL_USER = None
    DESTINATION_EMAIL_LIST = []

MAIA_BOUNDARY_URL = "https://baze.cm-maia.pt/BaZe/api/api4gj.php?nome=limconcvm"
OFFICIAL_DATA_URL = "https://geoserver.cm-maia.pt/geoserver/amp/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=amp:ugsSaude&maxFeatures=5000&outputFormat=application%2Fjson"

HTML_MAP_FILENAME = os.path.join(DRIVE_SAVE_PATH, f"mapa_saude_v74_{datetime.date.today().isoformat()}.html")
GOOGLE_SHEET_NAME = f"Reconciliacao_Saude_V74_{datetime.date.today().isoformat()}"

OVERPASS_URLS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]
API_TIMEOUT = 180
MATCH_THRESHOLD = 300
DIST_LIMIT_STRICT = 30.0

# --- ESTILOS VISUAIS (Igual ao Social) ---
STYLE_CONFIG = {
    "Total": {"folium": "green", "hex": "green", "icon": "star", "label": "Match Perfeito (3 Fontes)"},
    "Parcial": {"folium": "lightgreen", "hex": "#90EE90", "icon": "check", "label": "Match Parcial (2 Fontes)"},

    "ConflitoLoc": { "folium": "orange", "hex": "#FFA500", "icon": "arrows-alt", "label": "Conflito Localização (>30m)" },
    "ConflitoNome": { "folium": "blue", "hex": "#1E90FF", "icon": "font", "label": "Conflito de Nomes" },
    "ConflitoMisto": { "folium": "red", "hex": "#FF0000", "icon": "bomb", "label": "Conflito Misto (Nome + Local)" },

    "Apenas": {"folium": "purple", "hex": "purple", "icon": "question", "label": "Apenas no WFS"},
    "Default": {"folium": "gray", "hex": "gray", "icon": "info", "label": "Indefinido"}
}

# --- 4. FUNÇÕES AUXILIARES & PROFILING ---
profiling_log = []
script_start_time = time.time()

def start_timer(): return time.time()

def log_time(task, start_t):
    elapsed_proc = time.time() - start_t
    elapsed_watch = time.time() - script_start_time
    print(f"[{task}] {elapsed_proc:.2f}s")
    profiling_log.append({"task": task, "proc_time": elapsed_proc, "watch_time": elapsed_watch})

def normalize_text(text):
    if not text: return ""
    return unidecode(str(text)).strip().lower()

def approx_distance_meters(lat1, lon1, lat2, lon2):
    try:
        if not all([lat1, lon1, lat2, lon2]): return float('inf')
        R = 6371000
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)
        a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    except: return float('inf')

def get_safe_bounds(poly):
    try:
        minx, miny, maxx, maxy = poly.bounds
        if np.isnan(minx): raise ValueError("Bounds NaN")
        return minx, miny, maxx, maxy
    except:
        return -8.691, 41.185, -8.530, 41.309

context_vars = {'sheet_url': '#', 'date': str(datetime.date.today())}

# --- 5. CARREGAMENTO ---

def load_maia_boundary(url):
    s = start_timer()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = requests.get(url, verify=False, timeout=30)
            gdf = gpd.read_file(BytesIO(r.content))
            if gdf.empty: return None
            if gdf.crs and gdf.crs.to_string() != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")
            poly = gdf.geometry.unary_union
            if not poly.is_valid: poly = poly.buffer(0)
            log_time("Limite Maia", s)
            return poly
    except Exception as e:
        print(f"Erro limite: {e}")
        return None

def query_osm_bulletproof(polygon, timeout_seconds, buffer, urls_list):
    s = start_timer()
    print("A carregar OSM (Tags Saúde)...")
    minx, miny, maxx, maxy = get_safe_bounds(polygon)
    bbox = f"{miny-buffer:.5f},{minx-buffer:.5f},{maxy+buffer:.5f},{maxx+buffer:.5f}"

    # Query adaptada para Saúde
    query = f"""
    [out:json][timeout:{timeout_seconds}];
    (
      nwr["amenity"="pharmacy"]({bbox});
      nwr["amenity"="hospital"]({bbox});
      nwr["amenity"="clinic"]({bbox});
      nwr["amenity"="doctors"]({bbox});
      nwr["amenity"="dentist"]({bbox});
      nwr["healthcare"]({bbox});
      nwr["health_specialty"]({bbox});
    );
    out center tags;
    """
    for url in urls_list:
        try:
            r = requests.post(url, data={"data": query}, timeout=timeout_seconds)
            if r.status_code == 200:
                data = r.json().get("elements", [])
                log_time(f"OSM Raw ({len(data)})", s)
                return data
            elif r.status_code == 429: time.sleep(2)
        except: continue
    print("❌ Falha Total no OSM.")
    return []

def process_osm_elements(elements):
    features = []
    if not elements: return gpd.GeoDataFrame()
    for e in elements:
        lat, lon = None, None
        if e['type'] == 'node': lat, lon = e.get("lat"), e.get("lon")
        elif "center" in e: lat, lon = e["center"]["lat"], e["center"]["lon"]
        if lat and lon:
            tags = e.get("tags", {})
            name = tags.get("name", "Sem Nome")
            features.append({"Name": name, "Name_Clean": normalize_text(name), "Latitude": lat, "Longitude": lon, "OSM_ID": e.get("id"), "geometry": Point(lon, lat)})
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def filter_points_safe(gdf, poly):
    if gdf.empty: return gdf
    if gdf.crs != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")
    try:
        filtered = gdf[gdf.within(poly)].copy()
        if len(filtered) == 0 and len(gdf) > 0: return gdf
        return filtered
    except: return gdf

def load_wfs(url):
    s = start_timer()
    print("A carregar WFS...")
    try:
        r = requests.get(url, verify=False, timeout=60)
        gdf = gpd.read_file(BytesIO(r.content))
        if gdf.crs is None: gdf.set_crs("EPSG:3763", inplace=True)
        if gdf.crs.to_string() != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")
        col = 'designacao' if 'designacao' in gdf.columns else 'nome'
        gdf['Name'] = gdf.get(col, 'Sem Nome')
        gdf['Name_Clean'] = gdf['Name'].apply(normalize_text)
        log_time(f"WFS ({len(gdf)})", s)
        return gdf
    except: return gpd.GeoDataFrame()

def load_overture_pcp(poly_geom):
    s = start_timer()
    print("A carregar Overture Maps (S3)...")
    try:
        minx, miny, maxx, maxy = get_safe_bounds(poly_geom)
        conn = duckdb.connect()
        conn.execute("INSTALL spatial; LOAD spatial;")
        conn.execute("INSTALL httpfs; LOAD httpfs;")
        conn.execute("SET s3_region='us-west-2';")
        conn.execute("SET s3_access_key_id=''; SET s3_secret_access_key='';")

        # Query adaptada para Categorias de SAÚDE
        query = f"""
        SELECT names.primary AS Name, categories.primary AS Category, ST_AsWKB(geometry) AS geometry
        FROM read_parquet('s3://overturemaps-us-west-2/release/2025-12-17.0/theme=places/type=place/*', hive_partitioning=1)
        WHERE bbox.xmin BETWEEN {minx - 0.05:.4f} AND {maxx + 0.05:.4f} AND bbox.ymin BETWEEN {miny - 0.05:.4f} AND {maxy + 0.05:.4f}
        AND (
            categories.primary IN ('pharmacy', 'hospital', 'medical_clinic', 'dentist', 'physical_therapy', 'medical_lab', 'doctor', 'health_care')
            OR names.primary ILIKE '%Farmácia%' OR names.primary ILIKE '%Clínica%' OR names.primary ILIKE '%Hospital%' OR names.primary ILIKE '%USF%' OR names.primary ILIKE '%UCSP%'
        )
        """
        df = conn.execute(query).fetchdf()
        if df.empty: return gpd.GeoDataFrame()
        df['geometry'] = df['geometry'].apply(lambda x: wkb.loads(bytes(x)) if x else None)
        gdf = gpd.GeoDataFrame(df.dropna(subset=['geometry']), geometry='geometry', crs="EPSG:4326")
        gdf['Name_Clean'] = gdf['Name'].apply(normalize_text)
        filtered = filter_points_safe(gdf, poly_geom)
        if not filtered.empty:
            log_time(f"Overture ({len(filtered)})", s)
            return filtered
        return gdf
    except Exception as e:
        print(f"❌ Erro Overture: {e}")
        return gpd.GeoDataFrame()

# --- 7. RECONCILIAÇÃO ---
def standardize_name_saude(text):
    if not isinstance(text, str): return ""
    t = unidecode(text).lower().strip()
    replacements = {
        r'\busf\b': 'unidade saude familiar',
        r'\bucsp\b': 'unidade cuidados saude',
        r'\bclinica\b': 'clinica',
        r'\bfarmacia\b': 'farmacia',
        r'\bdr\.\b': '',
        r'\bdra\.\b': '',
        r'\bunidade local de saude\b': 'uls',
    }
    for pattern, repl in replacements.items(): t = re.sub(pattern, repl, t)
    return t

def check_incompatible_categories_saude(name1, name2):
    n1, n2 = normalize_text(name1), normalize_text(name2)
    cats = {
        'farmacia': ['hospital', 'clinica', 'dentista', 'analises'],
        'dentista': ['farmacia', 'hospital', 'fisioterapia'],
        'analises': ['farmacia', 'dentista'],
    }
    for key, forbidden_list in cats.items():
        if key in n1:
            for forbidden in forbidden_list:
                if forbidden in n2: return True
        if key in n2:
            for forbidden in forbidden_list:
                if forbidden in n1: return True
    return False

def reconcile_final(wfs, osm, overture):
    s = start_timer()
    print("A reconciliar (Saúde)...")
    results = []

    if not overture.empty:
        overture['lat'] = overture.geometry.y
        overture['lon'] = overture.geometry.x

    def find_best_match(target, candidates, source_name):
        if candidates.empty: return None, None, None, None
        df = candidates.copy()
        df['dist'] = df.geometry.apply(lambda g: approx_distance_meters(target.geometry.y, target.geometry.x, g.y, g.x))
        candidates_near = df[df['dist'] <= 700].copy()
        if candidates_near.empty: return None, None, None, None

        best_match, best_score = None, 0
        wfs_std = standardize_name_saude(target['Name'])

        for _, row in candidates_near.iterrows():
            cand_std = standardize_name_saude(row['Name'])
            dist = row['dist']
            if check_incompatible_categories_saude(target['Name'], row['Name']): continue
            base_score = fuzz.token_set_ratio(wfs_std, cand_std)
            final_score = base_score

            if dist > 400:
                if base_score < 90: final_score = 0
            elif dist > 100:
                if base_score < 80: final_score = 0
            if dist > DIST_LIMIT_STRICT: final_score -= (dist / 100)

            if final_score > best_score:
                best_score = final_score
                best_match = row

        if best_match is not None and best_score > 55:
            comment = None
            raw_dist = best_match['dist']
            raw_name = best_match['Name']
            match_score_real = fuzz.token_set_ratio(wfs_std, standardize_name_saude(raw_name))

            if raw_dist > DIST_LIMIT_STRICT:
                comment = f"[{source_name}] Mudou de local? (Dist: {int(raw_dist)}m)"
            elif match_score_real < 85:
                comment = f"[{source_name}] Mudou de nome? ('{raw_name}')"

            return raw_name, raw_dist, best_match.geometry, comment
        return None, None, None, None

    for _, w in wfs.iterrows():
        o_name, o_dist, o_geom, o_comm = find_best_match(w, osm, "OSM")
        v_name, v_dist, v_geom, v_comm = find_best_match(w, overture, "OV")

        matches = 1
        comments = []
        if o_name: matches += 1
        if v_name: matches += 1

        if o_comm: comments.append(o_comm)
        if v_comm: comments.append(v_comm)

        if matches == 3: code, status = "Total", "Match Perfeito (3 Fontes)"
        elif matches == 2: code, status = "Parcial", "Match Parcial (2 Fontes)"
        else: code, status = "Apenas", "Apenas no WFS"

        final_comment = " | ".join(comments) if comments else ""

        has_loc_issue = "Mudou de local" in final_comment
        has_name_issue = "Mudou de nome" in final_comment

        if has_loc_issue and has_name_issue:
            code, status = "ConflitoMisto", "⚠️ Conflito Misto (Nome + Local)"
        elif has_loc_issue:
            code, status = "ConflitoLoc", "⚠️ Conflito de Localização"
        elif has_name_issue:
            code, status = "ConflitoNome", "⚠️ Conflito de Nomes"

        def gc(geo): return (geo.y, geo.x) if geo else (None, None)
        def fmt(val): return f"{round(val, 2)}" if val is not None else "-"

        results.append({
            'Nome_Oficial_WFS': w['Name'], 'Status': status, 'Code': code, 'Comentarios': final_comment,
            'Nome_OSM': o_name or 'N/A', 'Dist_OSM': fmt(o_dist),
            'Nome_Overture': v_name or 'N/A', 'Dist_OV': fmt(v_dist),
            'Lat_Oficial': w.geometry.y, 'Lon_Oficial': w.geometry.x,
            'Lat_OSM': gc(o_geom)[0], 'Lon_OSM': gc(o_geom)[1],
            'Lat_OV': gc(v_geom)[0], 'Lon_OV': gc(v_geom)[1]
        })

    log_time("Reconciliação Semântica", s)
    return pd.DataFrame(results)

# --- 8. MAPA ---
class MapLegend(MacroElement):
    _template = Template(u"""
    {% macro html(this, kwargs) %}
    <div style="position: fixed; bottom: 30px; right: 30px; width: 340px; z-index:9999; border: 2px solid rgba(0,0,0,0.2); border-radius: 8px; background-color: rgba(255, 255, 255, 0.95); padding: 15px; font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

    <h4 style="margin-top:0; margin-bottom: 12px; font-size: 16px; color: #333; border-bottom: 1px solid #ddd; padding-bottom: 8px;"><b>{{ this.title }}</b></h4>

    <table style="width:100%; border-collapse: collapse;">
        {% for label, config in this.items %}
        <tr style="margin-bottom: 8px; border-bottom: 1px solid #f0f0f0;">
            <td style="width: 30px; text-align: center; padding: 6px;"><i class="fa fa-{{ config.icon }}" style="color: {{ config.hex }}; font-size: 16px;"></i></td>
            <td style="padding: 6px; color: #444;">{{ label }}</td>
        </tr>
        {% endfor %}
    </table>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px;">
        <small style="color:#555; font-weight:bold;">Linhas de Desvio (Conflitos):</small>
        <div style="font-size: 11px; margin-top:4px;">
            <span style="color:blue; font-weight:bold; font-size: 14px;">---</span> OSM
            <span style="margin-left:8px; color:cyan; font-weight:bold; font-size: 14px;">---</span> Overture
        </div>
    </div>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px; text-align: center;">
        <small style="color:#555; font-weight:bold; display:block; margin-bottom:4px;">Fontes de Dados</small>
        <div style="font-size: 12px; font-weight: bold; color: #444;">
            <a href="{{ this.wfs_url }}" target="_blank" style="color:#0078A8; text-decoration:none;">WFS</a>
            <span style="color:#ccc; margin:0 4px;">|</span>
            <a href="https://www.openstreetmap.org/" target="_blank" style="color:#0078A8; text-decoration:none;">OSM</a>
            <span style="color:#ccc; margin:0 4px;">|</span>
            <a href="https://overturemaps.org/" target="_blank" style="color:#0078A8; text-decoration:none;">OV</a>
        </div>
    </div>

    </div>
    {% endmacro %}
    """)
    def __init__(self, title, items, wfs_url):
        super(MapLegend, self).__init__()
        self._name = 'MapLegend'
        self.title = title
        self.items = items
        self.wfs_url = wfs_url

def create_map(gdf, poly_geom, filename):
    print("Gerando Mapa...")
    m = folium.Map(location=[41.23, -8.62], zoom_start=13, tiles="CartoDB positron")
    if poly_geom is not None:
        try:
            folium.GeoJson(data=mapping(poly_geom.simplify(0.0001)), name="Limite Maia", style_function=lambda x: {'color': '#000000', 'weight': 3, 'fillOpacity': 0, 'dashArray': '5, 5'}).add_to(m)
        except: pass

    def is_valid(lat, lon):
        try:
            if lat is None or lon is None: return False
            if math.isnan(float(lat)) or math.isnan(float(lon)): return False
            return True
        except: return False

    if not gdf.empty:
        for _, r in gdf.iterrows():
            if not is_valid(r['Lat_Oficial'], r['Lon_Oficial']): continue
            code = r['Code']
            config = STYLE_CONFIG.get(code, STYLE_CONFIG['Default'])
            comment_html = f"<div style='margin-top:5px; padding: 6px; background-color: #ffebee; border: 1px solid #ffcdd2; color: #b71c1c; font-size: 11px;'>⚠️ {r['Comentarios']}</div>" if r['Comentarios'] else ""

            popup_html = f"""
            <div style='font-family: "Segoe UI", sans-serif; width: 380px; font-size: 13px;'>
                <div style='background-color: {config['hex']}; color: {'white' if 'Conflito' in code else 'black'}; padding: 6px 10px; border-radius: 4px;'><b>{r['Status']}</b></div>
                {comment_html}
                <div style="margin-top: 8px;"><b style='color:#333; font-size:15px;'>⚕️ {r['Nome_Oficial_WFS']}</b></div>
                <div style='margin-top:8px; padding: 8px; background-color: #f0f7fb; border-left: 3px solid #0078A8;'><b>📍 Coords WFS:</b> {r['Lat_Oficial']:.5f}, {r['Lon_Oficial']:.5f}</div>
                <hr>
                <table style='width:100%; font-size:12px;'>
                    <tr><td><b>🌍 OSM:</b></td><td>{r['Nome_OSM']}</td><td>{r['Dist_OSM']}m</td></tr>
                    <tr><td><b>🏢 OV:</b></td><td>{r['Nome_Overture']}</td><td>{r['Dist_OV']}m</td></tr>
                </table>
            </div>"""

            folium.Marker([r['Lat_Oficial'], r['Lon_Oficial']], popup=folium.Popup(popup_html, max_width=400), icon=folium.Icon(color=config['folium'], icon=config['icon'], prefix='fa')).add_to(m)

            if is_valid(r['Lat_OSM'], r['Lon_OSM']): folium.PolyLine([[r['Lat_Oficial'], r['Lon_Oficial']], [r['Lat_OSM'], r['Lon_OSM']]], color="blue", weight=2, dash_array="5,5").add_to(m)
            if is_valid(r['Lat_OV'], r['Lon_OV']): folium.PolyLine([[r['Lat_Oficial'], r['Lon_Oficial']], [r['Lat_OV'], r['Lon_OV']]], color="cyan", weight=2, dash_array="5,5").add_to(m)

    items = [(v['label'], v) for k, v in STYLE_CONFIG.items() if k != "Default"]
    m.add_child(MapLegend("Auditoria Saúde", items, OFFICIAL_DATA_URL))
    m.save(filename)
    return m

def save_sheet(df, name):
    try:
        colab_auth.authenticate_user(); creds, _ = google_auth_default(); gc = gspread.authorize(creds)
        try: sh = gc.open(name)
        except: sh = gc.create(name)
        ws = sh.get_worksheet(0); ws.clear()
        df_save = df.drop(columns=['geometry'], errors='ignore')
        ws.update([df_save.columns.values.tolist()] + df_save.fillna('').astype(str).values.tolist(), 'A1')
        for e in DESTINATION_EMAIL_LIST:
            try: sh.share(e, perm_type='user', role='writer')
            except: pass
        return f"https://docs.google.com/spreadsheets/d/{sh.id}"
    except: return "#"

# --- 9. FUNÇÃO DE EMAIL ---
def send_execution_email(sheet_url):
    if not GMAIL_USER or not DESTINATION_EMAIL_LIST:
        print("⏭️ Email não enviado (Credenciais em falta).")
        return

    print("📧 A enviar email de relatório...")

    log_rows = ""
    for entry in profiling_log:
        log_rows += f"""
        <tr>
            <td style="border: 1px solid black; padding: 4px; font-family: monospace; white-space: nowrap;">{entry['task']}</td>
            <td style="border: 1px solid black; padding: 4px; text-align: right; font-family: monospace;">{entry['watch_time']:.2f}</td>
            <td style="border: 1px solid black; padding: 4px; text-align: right; font-family: monospace;">{entry['proc_time']:.2f}</td>
        </tr>
        """

    total_time = time.time() - script_start_time

    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2 style="color: #2E86C1;">Pipeline Saúde - Execution Log</h2>
        <p>O processo de reconciliação de dados de SAÚDE foi concluído.</p>

        <p><b>🔗 Google Sheet Resultado:</b> <a href="{sheet_url}">{sheet_url}</a></p>

        <h3 style="margin-bottom: 5px; margin-top:20px;">Execution Timing Log</h3>

        <table style="border-collapse: collapse; width: 100%; font-size: 12px; border: 2px solid black; font-family: Arial, sans-serif;">
            <thead style="background-color: #f2f2f2;">
                <tr>
                    <th style="border: 1px solid black; padding: 6px; text-align:left;">Task(s)</th>
                    <th style="border: 1px solid black; padding: 6px; text-align: right;">watch time (secs)</th>
                    <th style="border: 1px solid black; padding: 6px; text-align: right;">proc time (secs)</th>
                </tr>
            </thead>
            <tbody>
                {log_rows}
            </tbody>
            <tfoot>
                <tr style="font-weight: bold; background-color: #e6e6e6;">
                    <td style="border: 1px solid black; padding: 6px;">Overall (before email):</td>
                    <td style="border: 1px solid black; padding: 6px; text-align: right;">{total_time:.2f}</td>
                    <td style="border: 1px solid black; padding: 6px; text-align: right;">{total_time:.2f}</td>
                </tr>
            </tfoot>
        </table>

        <p style="font-size: 11px; color: #777; margin-top: 20px;">Automated via Google Colab Pipeline.</p>
    </body>
    </html>
    """

    msg = MIMEMultipart()
    msg['From'] = GMAIL_USER
    msg['To'] = ", ".join(DESTINATION_EMAIL_LIST)
    msg['Subject'] = f"Pipeline Log: Maia Saúde {datetime.date.today()}"
    msg.attach(MIMEText(html_body, 'html'))

    try:
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
        server.sendmail(GMAIL_USER, DESTINATION_EMAIL_LIST, msg.as_string())
        server.quit()
        print("✅ Email enviado com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao enviar email: {e}")

# --- EXECUÇÃO ---
print("\n--- INICIO PIPELINE SAÚDE---")
maia_geom = load_maia_boundary(MAIA_BOUNDARY_URL)

if maia_geom is not None:
    wfs = load_wfs(OFFICIAL_DATA_URL)
    osm_elements = query_osm_bulletproof(maia_geom, API_TIMEOUT, 0.02, OVERPASS_URLS)
    osm = process_osm_elements(osm_elements)
    osm = filter_points_safe(osm, maia_geom)

    ov = load_overture_pcp(maia_geom)

    df_res = reconcile_final(wfs, osm, ov)

    if not df_res.empty:
        url = save_sheet(df_res, GOOGLE_SHEET_NAME)
        context_vars['sheet_url'] = url
        m = create_map(df_res, maia_geom, HTML_MAP_FILENAME)
        print(f"✅ Mapa gerado: {HTML_MAP_FILENAME}")

        send_execution_email(url)
    else: print("❌ Erro: Tabela vazia.")
else: print("❌ Erro: Limite Maia.")

Instalando bibliotecas...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompat

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Overture (1657)] 7.15s
A reconciliar (Saúde)...
[Reconciliação Semântica] 3.74s
Gerando Mapa...
✅ Mapa gerado: /content/mapa_saude_v74_2026-01-21.html
📧 A enviar email de relatório...
✅ Email enviado com sucesso!


In [ ]:
display(m)